# LLM runtime benchmark (generative models only)

Measures wall-clock runtime for **decoder-only** cybersecurity LLMs on one inference sample.
Encoder models (SecureBERT, CySecBERT, CTI-BERT, etc.) are evaluated in **Score.ipynb** / **Plan.ipynb**, not here.

Configure model IDs via environment variables (see `utils/llm_benchmark.py`).


In [1]:
# Imports
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from utils.llm_benchmark import (
    DEFAULT_GENERATION_KWARGS,
    build_prompt_bundle,
    completed_runtime_frame,
    display_result,
    load_latest_inference_samples,
    model_support_table,
    results_to_frame,
    run_model_benchmark,
    sample_overview,
    select_benchmark_sample,
)

pd.set_option("display.max_colwidth", 140)
plt.style.use("ggplot")

## Load one inference sample

This cell loads the newest per-class prediction JSON under `output/inference/<CLASS>/`, picks one sample, and builds the shared prompt that every model cell will reuse.

Change `selected_label`, `selected_sample_id`, or `selected_index` if you want to benchmark a different inference row.

In [2]:
inference_samples = load_latest_inference_samples(verbose=True)

# Change these selectors if you want a different benchmark input.
selected_label = None
selected_sample_id = None
selected_index = 0

benchmark_sample = select_benchmark_sample(
    inference_samples,
    predicted_label=selected_label,
    sample_id=selected_sample_id,
    index=selected_index,
)
prompt_bundle = build_prompt_bundle(benchmark_sample)

print("Selected sample overview:")
display(pd.DataFrame([sample_overview(benchmark_sample)]))

print("Model support / execution notes:")
display(pd.DataFrame(model_support_table()))

print("Prompt preview:\n")
print(prompt_bundle["user_prompt"][:4000])

Latest inference files:
  [BENIGN] predictions_20260521_124653.json
  [DDOS] predictions_20260521_124653.json
  [DOS] predictions_20260521_124653.json
  [PORTSCAN] predictions_20260521_124653.json
  [FTPPATATOR] predictions_20260521_124653.json
Loaded 250 sample(s) from 5 class file(s).
Selected sample overview:


,sample_id,predicted_label,confidence,dominant_agent,dominant_contribution_pct,source_file,source_label_dir
0,0,BENIGN,0.999733,FlowProbe,0.823984,predictions_20260521_124653.json,BENIGN


Model support / execution notes:


,model_name,backend,note,model_id_env,api_model_env
0,SecureBERT,unsupported,Encoder-only cybersecurity BERT; no direct text-generation interface.,NaN,NaN
1,DarkBERT,unsupported,"Encoder-only domain-adapted BERT; use for embeddings/classification, not generation.",NaN,NaN
2,CySecBERT,unsupported,Encoder-style cybersecurity BERT; not a standalone causal text generator.,NaN,NaN
3,CTI-BERT,unsupported,Encoder-only CTI model; no direct free-form generation path.,NaN,NaN
4,CyBERT,unsupported,BERT-style encoder family; unsupported for direct text generation in this benchmark.,NaN,NaN
5,Foundation-Sec-8B,hf_causal,Attempts local Hugging Face causal generation when the repo id/path is configured.,FOUNDATION_SEC_8B_MODEL_ID,NaN
6,Foundation-Sec-8B-Reasoning,hf_causal,Attempts local Hugging Face causal generation when the repo id/path is configured.,FOUNDATION_SEC_8B_REASONING_MODEL_ID,NaN
7,SecGPT,mixed_api_or_hf,"Uses an OpenAI-compatible API when SECGPT_API_* vars are set, otherwise falls back to a local Hugging Face causal model when SECGPT_MODE...",SECGPT_MODEL_ID,SECGPT_API_MODEL
8,CyberInstruct,mixed_api_or_hf,"Uses an OpenAI-compatible API when CYBERINSTRUCT_API_* vars are set, otherwise falls back to a local Hugging Face causal model when CYBE...",CYBERINSTRUCT_MODEL_ID,CYBERINSTRUCT_API_MODEL
9,PentestGPT,unsupported,PentestGPT is a framework/workflow rather than one standalone model checkpoint.,NaN,NaN


Prompt preview:

You are a cybersecurity advisor for a multi-device NIDS deployment.
        Your role is to recommend practical response steps at each monitoring/enforcement point.
        You may suggest any reasonable mitigations; you are not limited to a fixed action vocabulary.

        INPUT CONTEXT

        Prediction summary:
        - sample_id: 0
        - predicted_label: BENIGN
        - confidence: 100.0%
- Dominant device for this prediction: FlowProbe (contribution: 82.4%)

        Inference evidence (condensed SHAP — top 3 features per device):
        {
  "predicted_label": "BENIGN",
  "confidence": 0.9997325539588928,
  "dominant_tier": "FlowProbe",
  "dominant_contribution_pct": 82.39843810070218,
  "tier_contributions_pct": {
    "FlowProbe": 82.39843810070218,
    "Firewall": 14.654215488948704,
    "CoreRouter": 2.9473464103491107
  },
  "top_features_by_tier": {
    "FlowProbe": [
      "udps.syn_packet_count",
      "udps.srcdst_syn_packet_count",
      "udps.sr

## Shared runtime configuration

Update `generation_kwargs` if you want longer generations or sampling. The helper module uses these environment variables for the models that can attempt generation:

- `FOUNDATION_SEC_8B_MODEL_ID`
- `FOUNDATION_SEC_8B_REASONING_MODEL_ID`
- `SECGPT_MODEL_ID` or `SECGPT_API_KEY` + `SECGPT_API_MODEL` (+ optional `SECGPT_API_BASE_URL`)
- `CYBERINSTRUCT_MODEL_ID` or `CYBERINSTRUCT_API_KEY` + `CYBERINSTRUCT_API_MODEL` (+ optional `CYBERINSTRUCT_API_BASE_URL`)

If those variables are not set, the model cell returns `needs_configuration` rather than failing silently.

In [3]:
generation_kwargs = DEFAULT_GENERATION_KWARGS.copy()
generation_kwargs.update(
    {
        "max_new_tokens": 320,
        "temperature": 0.2,
        "top_p": 0.9,
        "do_sample": False,
    }
)

benchmark_results = {}
print("Generation kwargs:")
print(generation_kwargs)

Generation kwargs:
{'max_new_tokens': 320, 'temperature': 0.2, 'top_p': 0.9, 'do_sample': False}


## Model cells

Run any subset of these cells, or run them top-to-bottom to populate `benchmark_results` for the final runtime comparison.

In [9]:
foundation_sec_8b_result = run_model_benchmark(
    "Foundation-Sec-8B",
    benchmark_sample,
    generation_kwargs=generation_kwargs,
)
benchmark_results["Foundation-Sec-8B"] = foundation_sec_8b_result
display_result(foundation_sec_8b_result)

Model: Foundation-Sec-8B
Status: needs_configuration
Backend used: hf_causal
Runtime (s): None
Error / note:
Set FOUNDATION_SEC_8B_MODEL_ID to a local model path or Hugging Face repo id for Foundation-Sec-8B.


In [10]:
foundation_sec_8b_reasoning_result = run_model_benchmark(
    "Foundation-Sec-8B-Reasoning",
    benchmark_sample,
    generation_kwargs=generation_kwargs,
)
benchmark_results["Foundation-Sec-8B-Reasoning"] = foundation_sec_8b_reasoning_result
display_result(foundation_sec_8b_reasoning_result)

Model: Foundation-Sec-8B-Reasoning
Status: needs_configuration
Backend used: hf_causal
Runtime (s): None
Error / note:
Set FOUNDATION_SEC_8B_REASONING_MODEL_ID to a local model path or Hugging Face repo id for Foundation-Sec-8B-Reasoning.


In [11]:
secgpt_result = run_model_benchmark(
    "SecGPT",
    benchmark_sample,
    generation_kwargs=generation_kwargs,
)
benchmark_results["SecGPT"] = secgpt_result
display_result(secgpt_result)

Model: SecGPT
Status: needs_configuration
Backend used: mixed_api_or_hf
Runtime (s): None
Error / note:
Set SECGPT_MODEL_ID to a local model path or Hugging Face repo id for SecGPT.


In [12]:
cyberinstruct_result = run_model_benchmark(
    "CyberInstruct",
    benchmark_sample,
    generation_kwargs=generation_kwargs,
)
benchmark_results["CyberInstruct"] = cyberinstruct_result
display_result(cyberinstruct_result)

Model: CyberInstruct
Status: needs_configuration
Backend used: mixed_api_or_hf
Runtime (s): None
Error / note:
Set CYBERINSTRUCT_MODEL_ID to a local model path or Hugging Face repo id for CyberInstruct.


## Runtime comparison

Run this after the model cells you care about. Unsupported or unconfigured models remain in the summary table, while the chart only plots completed generations.

In [14]:
all_results = list(benchmark_results.values())
summary_df = results_to_frame(all_results)
completed_df = completed_runtime_frame(all_results)

print("All benchmark results:")
display(summary_df)

if completed_df.empty:
    print("No completed model runs yet. Configure at least one supported generator and run its cell first.")
else:
    ax = completed_df.plot(
        kind="bar",
        x="model_name",
        y="runtime_seconds",
        figsize=(12, 5),
        legend=False,
        color="#4C72B0",
        title="LLM Runtime Comparison (completed runs only)",
    )
    ax.set_xlabel("Model")
    ax.set_ylabel("Runtime (seconds)")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

All benchmark results:


,model_name,status,backend_used,runtime_seconds,predicted_label,sample_id,error
0,Foundation-Sec-8B,needs_configuration,hf_causal,NaN,BENIGN,0,Set FOUNDATION_SEC_8B_MODEL_ID to a local model path or Hugging Face repo id for Foundation-Sec-8B.
1,Foundation-Sec-8B-Reasoning,needs_configuration,hf_causal,NaN,BENIGN,0,Set FOUNDATION_SEC_8B_REASONING_MODEL_ID to a local model path or Hugging Face repo id for Foundation-Sec-8B-Reasoning.
2,SecGPT,needs_configuration,mixed_api_or_hf,NaN,BENIGN,0,Set SECGPT_MODEL_ID to a local model path or Hugging Face repo id for SecGPT.
3,CyberInstruct,needs_configuration,mixed_api_or_hf,NaN,BENIGN,0,Set CYBERINSTRUCT_MODEL_ID to a local model path or Hugging Face repo id for CyberInstruct.
4,SecureBERT,unsupported,unsupported,0.0,BENIGN,0,
5,DarkBERT,unsupported,unsupported,0.0,BENIGN,0,
6,CySecBERT,unsupported,unsupported,0.0,BENIGN,0,
7,CTI-BERT,unsupported,unsupported,0.0,BENIGN,0,
8,CyBERT,unsupported,unsupported,0.0,BENIGN,0,
9,PentestGPT,unsupported,unsupported,0.0,BENIGN,0,


No completed model runs yet. Configure at least one supported generator and run its cell first.
